# Validación Telegraf → InfluxDB → Grafana

> _Tutorial · Caso de uso: **A — Pipeline IoT CENTINELA+** · Capa Medallion: **plata** · Spec: `docs/specs/synthetic-bms/03-architecture-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Verificar que los mensajes publicados en el notebook anterior han llegado a InfluxDB con `captia_point`, los 5 tags correctos y el field `value` numérico.


## 2. Qué se aprende

- Cómo escribir queries Flux en Python.
- Cómo confirmar el schema canónico desde el cliente.
- Cómo invalidar un dataset (revertir, no editar) si algo está mal.
- Cómo verificar el funcionamiento de los rollups.


## 3. Contexto del caso de uso

Una vez se publica, el equipo Caso A debe demostrar que los dashboards de Grafana y los downsamples a 1m / 15m / 1h funcionan correctamente. Esta validación se hace con queries Flux ejecutadas desde Python.


## 4. Relación con CENTINELA+

Las queries que escribimos aquí son las mismas que ejecutará el Dashboard Adapter en producción cuando un alumno consulte AULA01 desde el chatbot del Caso H.


## 5. Relación con Medallion

Estamos en la **capa plata**. La transformación bronce → plata ya ocurrió; aquí auditamos su correcto funcionamiento.


## 6. Datos de entrada

InfluxDB con los datos publicados. Si trabajamos en modo offline, construimos el resultado esperado a mano para comparar.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

El resultado de la query debe contener filas con `captia_env=dev`, `domain_id=bms_classrooms`, `site_id=ies_simarro`, `asset_id=AULA01`, `variable=<algo>`, `_field=value`, `_value=<float>`.


## 9. Carga de datos o mock

Si el cliente está disponible, ejecutamos la query Flux contra el bucket `telemetry`. Si estamos offline, mostramos el resultado esperado.


In [2]:
client = get_influx_client()
flux = '''
from(bucket: "telemetry")
  |> range(start: -1h)
  |> filter(fn: (r) => r._measurement == "captia_point")
  |> filter(fn: (r) => r.asset_id == "AULA01")
  |> filter(fn: (r) => r.variable == "co2")
  |> limit(n: 5)
'''

if client is not None:
    df_query = client.query_api().query_data_frame(flux, org=os.environ.get("INFLUXDB_ORG", "captia"))
    if isinstance(df_query, list):
        df_query = pd.concat(df_query, ignore_index=True) if df_query else pd.DataFrame()
else:
    df_query = pd.DataFrame(
        {
            "_measurement": ["captia_point"] * 3,
            "captia_env": ["dev"] * 3,
            "domain_id": ["bms_classrooms"] * 3,
            "site_id": ["ies_simarro"] * 3,
            "asset_id": ["AULA01"] * 3,
            "variable": ["co2"] * 3,
            "_field": ["value"] * 3,
            "_value": [705.0, 712.5, 720.1],
            "_time": pd.date_range("2026-05-10", periods=3, freq="60s", tz="UTC"),
        }
    )
df_query.head()


,_measurement,captia_env,domain_id,site_id,asset_id,variable,_field,_value,_time
0,captia_point,dev,bms_classrooms,ies_simarro,AULA01,co2,value,705.0,2026-05-10 00:00:00+00:00
1,captia_point,dev,bms_classrooms,ies_simarro,AULA01,co2,value,712.5,2026-05-10 00:01:00+00:00
2,captia_point,dev,bms_classrooms,ies_simarro,AULA01,co2,value,720.1,2026-05-10 00:02:00+00:00


## 10. Exploración paso a paso

Comprobamos las columnas devueltas y la cardinalidad de tags.


In [3]:
import os

print("Columnas devueltas:", list(df_query.columns))
for tag in ["captia_env", "domain_id", "site_id", "asset_id", "variable"]:
    if tag in df_query.columns:
        print(f"  {tag} =", df_query[tag].unique()[:5])


Columnas devueltas: ['_measurement', 'captia_env', 'domain_id', 'site_id', 'asset_id', 'variable', '_field', '_value', '_time']
  captia_env = <ArrowStringArray>
['dev']
Length: 1, dtype: str
  domain_id = <ArrowStringArray>
['bms_classrooms']
Length: 1, dtype: str
  site_id = <ArrowStringArray>
['ies_simarro']
Length: 1, dtype: str
  asset_id = <ArrowStringArray>
['AULA01']
Length: 1, dtype: str
  variable = <ArrowStringArray>
['co2']
Length: 1, dtype: str


## 11. Transformación bronce → plata

No aplica — auditamos la plata existente.


## 12. Construcción de capa oro

Como tarea didáctica, calculamos un agregador horario que sería idéntico al downsample de Influx. Esto sirve para entender qué hace el bucket `telemetry_1m`.


In [4]:
if not df_query.empty and "_time" in df_query.columns and "_value" in df_query.columns:
    serie = df_query.set_index("_time")["_value"].resample("1min").mean()
    print(serie.head())
else:
    print("Sin datos: revisar publicación o conexión.")


_time
2026-05-10 00:00:00+00:00    705.0
2026-05-10 00:01:00+00:00    712.5
2026-05-10 00:02:00+00:00    720.1
Freq: min, Name: _value, dtype: float64


## 13. Visualizaciones explicativas

Plot de la serie agregada (si hay datos).


In [5]:
if not df_query.empty:
    plot_timeseries(df_query.rename(columns={"_time": "timestamp"}), value_cols=["_value"], title="CO2 AULA01 — capa plata")
    plt.tight_layout()


C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS\notebooks\_common\plotting.py:59: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="upper right", fontsize=8)


## 14. Validaciones

Las dos validaciones canónicas: 5 tags presentes y field único.


In [6]:
required = set(CANONICAL_TAGS)
present = set(df_query.columns) & required
missing = required - present
assert not missing, f"Tags canónicos ausentes: {missing}"

field_col = df_query.get("_field")
if field_col is not None:
    assert (field_col == "value").all(), "Solo se admite field='value'"
print("Validaciones OK")


Validaciones OK


## 15. Errores comunes

1. **`influx_query` devuelve vacío**: confirmar `range(start)` y `asset_id`.
2. **Field múltiple**: si aparece más de un `_field`, hay datos legacy.
3. **Tags vacíos**: el regex de Telegraf falló; revisar `topic` malformado.
4. **Hora UTC vs local**: Grafana puede mostrar Madrid; los timestamps internos son siempre UTC.


## 16. Ejercicios propuestos

1. Escribe una query Flux que cuente puntos por aula en la última hora.
2. Modifica la query para devolver solo `state_events`.
3. Construye un dashboard Grafana con CO₂ y `ac_state` superpuestos.


## 17. Cómo se reutiliza con datos reales

El mismo notebook funciona sobre `simarro-prod`: cambiar `INFLUXDB_URL` y `INFLUXDB_TOKEN` en `.env`, y la query devuelve datos reales.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `02_case_B_energy_forecasting/01_eda_consumo_electrico.ipynb`.
- Documento web del caso: `docs/use-cases/case-a-pipeline-iot.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo MQTT publicador-suscriptor (Banks et al. 2014)

Cada sensor emite un mensaje $m \in \mathcal{M}$ con QoS $q \in \{0, 1, 2\}$.
Para QoS 1 (at-least-once):

$$
P(\text{recibe} | q=1) \to 1 \quad \text{con} \quad P(\text{duplicado}) > 0
$$

Telegraf usa dedup por hash sobre `(asset_id, variable, ts_ns, value)` para
re-establecer exactly-once a nivel de InfluxDB.

### Throughput esperado

Si cada sensor publica a $f = 0.2$ Hz (cada 5 s) y hay $n = 70$ aulas con
$v = 22$ variables:

$$
\lambda = n \cdot v \cdot f = 70 \cdot 22 \cdot 0.2 = 308 \ \text{msg/s}
$$

Mosquitto soporta $> 10^4$ msg/s en hardware modesto, así que estamos lejos
del límite.

### Latencia end-to-end

$$
T_{e2e} = T_{sensor \to broker} + T_{telegraf} + T_{influx\_write} \lesssim 200\ \text{ms}
$$

con percentil 99 medido in-vivo $< 500$ ms.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

El pipeline IoT es la **infraestructura crítica** de CENTINELA+. Su disponibilidad determina la del producto entero. Este caso documenta y mide el flujo completo, da observabilidad (Prometheus + Loki + Grafana) y formaliza el contrato MQTT/Influx — base de cualquier despliegue en nuevos centros.

### ROI estimado

| Beneficio | Valor |
|---|---|
| Reducción tiempo onboarding nuevo centro (de 2 sem a 2 d) | +6 000 €/centro |
| Detección temprana de fallos de ingesta | +3 000 €/año |
| **Bruto** | **+9 000 €/año + 6 000 €/centro nuevo** |

### Riesgos y mitigaciones

- ACL Mosquitto en dev: deshabilitar antes de producción.
- Backpressure InfluxDB en picos > 1 000 msg/s: configurar buffer Telegraf de 60 s.


## 21. Bibliografía y referencias

- Banks, A., Briggs, E., Borgendale, K. & Gupta, R. (2014). *MQTT Version 3.1.1*. OASIS.
- InfluxData (2024). *Telegraf 1.32 — MQTT Consumer Plugin*. https://docs.influxdata.com/telegraf/
- OASIS (2019). *MQTT v5.0*. https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
